# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR² dataset on rangeland management adoption predictors using the `mlcroissant` library.

### Dataset Source
The dataset schema is available at [https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json), defined by the [Croissant](https://mlcommons.org/croissant/) standard.

In [ ]:
# Ensure `mlcroissant` is installed!pip install --quiet mlcroissant

## 1. Data Loading

Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{getattr(metadata, 'name', 'Unnamed Dataset')}: {getattr(metadata, 'description', '')}")

## 2. Data Overview

List the available record sets along with their `@id`, fields, and columns. These are referenced throughout the notebook using their `@id` as required.

In [ ]:
# Retrieve all record sets
record_sets = list(dataset.record_sets)

if not record_sets:
    print("No record sets found in the metadata. Please check the dataset schema or the loading method.")
else:
    for record_set in record_sets:
        print(f"RecordSet @id: {record_set.id}")
        print(f"  name: {getattr(record_set, 'name', '')}")
        print(f"  description: {getattr(record_set, 'description', '')}")
        print(f"  Fields:")
        for field in record_set.fields:
            print(f"    Field @id: {field.id}")
            print(f"      name: {getattr(field, 'name', '')}")
            if hasattr(field, 'columns'):
                for column in field.columns:
                    print(f"        Column @id: {column.id}, name: {getattr(column, 'name', '')}")
        print('-'*40)

## 3. Data Extraction

Load data from a specific record set into a DataFrame for analysis. Use entities' `@id` for referencing and extraction.

In [ ]:
# List all available record set @ids
record_set_ids = [r.id for r in dataset.record_sets]
print("Available RecordSet @ids:")
for rid in record_set_ids:
    print(f"  - {rid}")

# If no record set available, skip data extraction
dataframes = {}
if not record_set_ids:
    print("No record sets discovered; cannot extract records.")
else:
    # For demonstration, extract all record sets (adjust as needed)
    for rs_id in record_set_ids:
        df = pd.DataFrame(dataset.records(record_set=rs_id))
        dataframes[rs_id] = df
        print(f"Loaded RecordSet @id: {rs_id}, Shape: {df.shape}")
    # Use the first available record set for subsequent analysis
    selected_record_set_id = record_set_ids[0]
    print(f"\nColumns for RecordSet @id '{selected_record_set_id}':")
    print(list(dataframes[selected_record_set_id].columns))
    display(dataframes[selected_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

Apply example data processing: filtering, normalization, grouping—always referencing fields/columns by their `@id`.

In [ ]:
# EDA only if data is available
if not dataframes or not selected_record_set_id or dataframes[selected_record_set_id].empty:
    print("No data available for EDA. Please ensure the dataset provides record sets and records.")
else:
    df = dataframes[selected_record_set_id]

    # Infer numeric fields from the DataFrame
    numeric_fields = df.select_dtypes(include=['number']).columns.tolist()
    print(f"Numeric fields detected: {numeric_fields}")
    if not numeric_fields:
        print("No numeric fields present to analyze/filter.")
    else:
        numeric_field = numeric_fields[0]
        print(f"Using numeric field (column @id): {numeric_field}")

        threshold = df[numeric_field].mean()  # Use mean as an example threshold
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with '{numeric_field}' > {threshold:.2f} (column @id):")
        display(filtered_df.head())

        # Normalization
        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized '{numeric_field}' for filtered records (columns @ids shown):")
        display(filtered_df[[numeric_field, norm_col]].head())

        # Try to group by a likely categorical field: pick the first object-type column
        group_fields = df.select_dtypes(include=['object', 'category']).columns.tolist()
        if group_fields:
            group_field = group_fields[0]
            print(f"Grouping by field (column @id): {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame(name=f"mean_{numeric_field}")
            print(f"Mean '{numeric_field}' by '{group_field}':")
            display(grouped_df.head())
        else:
            print("No suitable categorical field found for grouping.")

## 5. Visualization

Visualize data distributions or relationships using the selected fields (by `@id`).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Only proceed if EDA set up 'filtered_df' and fields
if 'filtered_df' in locals() and not filtered_df.empty and 'numeric_field' in locals():
    plt.figure(figsize=(8, 4))
    sns.histplot(filtered_df[numeric_field], bins=25, kde=True)
    plt.xlabel(f"{numeric_field} (@id)")
    plt.title(f"Distribution of '{numeric_field}' for Filtered Records")
    plt.show()

    # Categorical grouping visualization if exists
    if 'group_field' in locals():
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=filtered_df[group_field], y=filtered_df[numeric_field])
        plt.xlabel(f"{group_field} (@id)")
        plt.ylabel(f"{numeric_field} (@id)")
        plt.title(f"'{numeric_field}' by Groups of '{group_field}'")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Insufficient data for visualization.")

## 6. Conclusion

- This notebook demonstrated how to use `mlcroissant` for loading, exploring, and analyzing a Croissant-defined FAIR² dataset.
- All dataset entities—record sets, fields, columns—were consistently referenced by their `@id` attributes.
- Further analysis can include advanced statistical modeling or integrating with policy informatics based on these predictors of knowledge adoption in rangeland management.
